# Load-Correlated Immunity Genes (Plates 1–5)

Updated version of the gene selection at the end of `Local_analysis/Followup_analyses_after_Luke_talk/Immunity_and_defense_score/immunity_metrics_with_non_self_recognition_gene_set.ipynb`. That notebook found 199 genes on plates 1–4.

Genes come from two sources:
1. **GO-term genes:** genes annotated to any of the defense GO terms in `Defense_go_terms.csv`, plus Luke's custom non-self-recognition set.
2. **FLG genes:** the Arabidopsis flg22-induced genes shared across species, from `shared_flg_induced_genes.csv`.

A gene is kept if its Pearson r with total bacterial load (`ham16s`) across long-term samples is > 0.4. The final list is the union of the two sources.

The transcriptome is the plates 1–5 set used in `bacteria_vs_transcriptome_time_abundance_rain_correlation_plates_1_to_5.ipynb`. Plate 5 re-preps replace the samples they were redone for, and for any remaining duplicated `sampID` the lower-read transcriptome is dropped.

In [1]:
import numpy as np
import pandas as pd

threshold = 0.4

## Defense GO-term and FLG gene sets

In [2]:
arabi_go_matrix = pd.read_csv(
    "/Users/michael/Data/Reference_data/Arabdidopsis_go_matrix_from_tair_gaf.csv",
    index_col=0,
)
defense_terms = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Defense_go_terms.csv"
)

# Luke's custom non-self-recognition gene set, added as its own "GO term"
non_self_recognition_genes = [
    "AT1G02920",
    "AT1G02930",
    "AT1G21110",
    "AT1G21120",
    "AT1G26380",
    "AT1G26410",
    "AT1G26420",
    "AT1G35230",
    "AT1G64170",
    "AT1G65500",
    "AT1G76930",
    "AT2G19190",
    "AT2G25470",
    "AT2G30750",
    "AT2G39200",
    "AT2G43620",
    "AT3G46280",
    "AT4G12490",
    "AT4G12500",
    "AT4G23140",
    "AT4G23220",
    "AT4G28420",
    "AT5G24110",
    "AT5G64120",
]
defense_terms.loc[len(defense_terms)] = [
    "GO:Luke Custom",
    "General Non Self Recognition Genes",
    "Custom From Luke",
]
arabi_go_matrix["GO:Luke Custom"] = 0
arabi_go_matrix.loc[non_self_recognition_genes, "GO:Luke Custom"] = 1

go_to_genes = {}
for go_term in defense_terms["GO Term ID"].unique():
    if go_term not in arabi_go_matrix.columns:
        print(f"Warning: GO term {go_term} not found in the matrix, skipping.")
        continue
    go_to_genes[go_term] = set(arabi_go_matrix.index[arabi_go_matrix[go_term] == 1])
    print(f"{go_term}: {len(go_to_genes[go_term])} genes")

go_term_genes = set().union(*go_to_genes.values())
print(f"\nUnique GO-term genes: {len(go_term_genes)}")

GO:0012501: 34 genes
GO:0009626: 54 genes
GO:0042742: 325 genes
GO:0071219: 7 genes
GO:0045087: 58 genes
GO:0009862: 16 genes
GO:0009861: 11 genes
GO:0034055: 1 genes
GO:0009682: 21 genes
GO:0009627: 57 genes
GO:0002237: 41 genes
GO:0010421: 2 genes
GO:0016045: 4 genes
GO:0009617: 153 genes
GO:1900425: 17 genes
GO:1900426: 30 genes
GO:1900424: 9 genes
GO:0002253: 1 genes
GO:0002218: 1 genes
GO:0002758: 13 genes
GO:0002757: 2 genes
GO:0006955: 15 genes
GO:0050776: 6 genes
GO:0009609: 2 genes
GO:Luke Custom: 24 genes

Unique GO-term genes: 663


In [3]:
flg_genes = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/shared_flg_induced_genes.csv"
)
flg_arabidopsis_genes = (
    flg_genes["Arabidopsis_thaliana"].dropna().astype(str).str.strip()
)
flg_arabidopsis_genes = set(
    gene for gene in flg_arabidopsis_genes if gene not in ["", "nan", "None"]
)
print(f"FLG Arabidopsis genes: {len(flg_arabidopsis_genes)}")

FLG Arabidopsis genes: 868


## Plates 1–5 transcriptome

In [4]:
transcriptome = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/plate1_5_tpm_normalized.csv",
    index_col=0,
)
rows_to_drop_expression_data = [
    "A2450525897_n01_undetermined",
    "A2449446903_n01_undetermined",
    "B250508004_n01_undetermined",
    "B2449500127_n01_undetermined",
]
transcriptome = transcriptome.drop(index=rows_to_drop_expression_data)
transcriptome = transcriptome.sort_index()

metadata = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Plates_1_to_5_metadata_merged_luke.csv",
    index_col=0,
)

## Get plate 5 samples and remove what they replaced from the metadata to avoid duplicates
plate_5_replacements = metadata.loc[metadata["rnaprepplate"] == "LICRNA_05"]
non_plate_5_samples = metadata.loc[metadata["rnaprepplate"] != "LICRNA_05"]
replaced_samples = non_plate_5_samples.loc[
    non_plate_5_samples["sampID"].isin(plate_5_replacements["sampID"])
]
trimmed_metadata = metadata.drop(index=replaced_samples.index)
### For remaining duplicated sampIDs, drop the transcriptome with fewer total reads
double_duplicates = (
    trimmed_metadata.loc[trimmed_metadata.duplicated(subset="sampID", keep=False)]
    .sort_values(by="Total Reads")
    .drop_duplicates(subset="sampID", keep="first")
)
trimmed_metadata = trimmed_metadata.drop(index=double_duplicates.index)
trimmed_transcriptome = transcriptome.loc[trimmed_metadata.index]

long_term_transcriptome = trimmed_transcriptome.loc[
    trimmed_metadata["Experiment Type"] == "Long Term"
]
long_term_metadata = trimmed_metadata.loc[
    trimmed_metadata["Experiment Type"] == "Long Term"
]
print("Long-term transcriptomes:", len(long_term_transcriptome))
print("Duplicated sampIDs remaining:", long_term_metadata["sampID"].duplicated().sum())

Long-term transcriptomes: 224
Duplicated sampIDs remaining: 0


In [5]:
total_load_df = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/Microbiome/lic2024_ham_16S_ITS.csv",
    index_col=0,
)
# Total load (ham16s) indexed by transcriptome filename
ham16s_by_filename = long_term_metadata["sampID"].map(total_load_df["ham16s"])
print("Long-term samples with ham16s:", ham16s_by_filename.notna().sum())

Long-term samples with ham16s: 222


## Genes with Pearson r > 0.4 vs total load

In [6]:
def load_correlated_genes(expression, load, threshold=threshold):
    """Pearson r of every GO-term / FLG gene with total load, and the genes over threshold."""
    candidate_genes = sorted((go_term_genes | flg_arabidopsis_genes) & set(expression.columns))
    pearson_with_load = expression[candidate_genes].corrwith(load)

    go_genes_over = set(
        pearson_with_load.index[
            pearson_with_load.index.isin(go_term_genes) & (pearson_with_load > threshold)
        ]
    )
    flg_genes_over = set(
        pearson_with_load.index[
            pearson_with_load.index.isin(flg_arabidopsis_genes)
            & (pearson_with_load > threshold)
        ]
    )
    return pearson_with_load, go_genes_over, flg_genes_over


pearson_correlation_with_load, go_term_genes_over_0_4_load, flg_genes_over_0_4_load = (
    load_correlated_genes(long_term_transcriptome, ham16s_by_filename)
)
shared_over_0_4_load = flg_genes_over_0_4_load & go_term_genes_over_0_4_load
union_over_0_4_load = flg_genes_over_0_4_load | go_term_genes_over_0_4_load

print(f"GO-term genes in transcriptome: {len(go_term_genes & set(long_term_transcriptome.columns))}")
print(f"FLG genes in transcriptome: {len(flg_arabidopsis_genes & set(long_term_transcriptome.columns))}")
print(f"\nFLG genes with Pearson r > {threshold}: {len(flg_genes_over_0_4_load)}")
print(f"GO-term genes with Pearson r > {threshold}: {len(go_term_genes_over_0_4_load)}")
print(f"Shared genes (both lists): {len(shared_over_0_4_load)}")
print(f"Union genes (either list): {len(union_over_0_4_load)}")

/Users/michael/miniconda3/envs/Luke_terrace/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/michael/miniconda3/envs/Luke_terrace/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


GO-term genes in transcriptome: 663
FLG genes in transcriptome: 868

FLG genes with Pearson r > 0.4: 63
GO-term genes with Pearson r > 0.4: 96
Shared genes (both lists): 11
Union genes (either list): 148


In [7]:
union_over_0_4_load_df = pd.DataFrame(
    {
        "gene": sorted(union_over_0_4_load),
        "pearson_r_load": pearson_correlation_with_load[sorted(union_over_0_4_load)].values,
    }
)
union_over_0_4_load_df["in_go_terms"] = union_over_0_4_load_df["gene"].isin(go_term_genes)
union_over_0_4_load_df["in_flg_genes"] = union_over_0_4_load_df["gene"].isin(
    flg_arabidopsis_genes
)
union_over_0_4_load_df = union_over_0_4_load_df.sort_values(
    "pearson_r_load", ascending=False
).reset_index(drop=True)
union_over_0_4_load_df

,gene,pearson_r_load,in_go_terms,in_flg_genes
0,AT1G26630,0.650972,True,False
1,AT3G01280,0.644043,True,False
2,AT2G03440,0.637452,True,False
3,AT5G15090,0.633388,True,False
4,AT4G29040,0.619435,True,False
...,...,...,...,...
143,AT4G39980,0.402448,True,True
144,AT4G35450,0.402308,True,False
145,AT3G18490,0.401966,True,False
146,AT3G52430,0.401910,True,False


In [8]:
union_over_0_4_load_df.to_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/load_correlated_genes_union_GO_and_FLG_plates_1_to_5.csv",
    index=False,
)

## Comparison with the plates 1–4 list

The original 199-gene list was never saved, so it is rebuilt here with the plates 1–4 setup from the original notebook: the plates 1–4 transcriptome with the unusable and borderline samples dropped.

In [9]:
transcriptome_1_4 = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/plate1_4_tpm_normalized.csv",
    index_col=0,
)
transcriptome_1_4 = transcriptome_1_4.drop(index=rows_to_drop_expression_data).sort_index()
metadata_1_4 = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Plates_1_to_4_metadata_merged_luke.csv",
    index_col=0,
)
unusable_samples = [
    "A2450525897_n01_LICRNA01_A06",
    "A2450525897_n01_LICRNA01_A08",
    "A2450525897_n01_LICRNA01_C11",
    "A2450525897_n01_LICRNA01_D11",
    "A2450525897_n01_LICRNA01_D12",
    "A2449446903_n01_LICRNA02_F01",
    "B2449500127_n01_LICRNA04_A04",
    "B2449500127_n01_LICRNA04_A06",
    "B2449500127_n01_LICRNA04_A07",
]
borderline_unusable = [
    "A2450525897_n01_LICRNA01_F03",
    "A2450525897_n01_LICRNA01_G07",
    "A2449446903_n01_LICRNA02_A04",
    "A2449446903_n01_LICRNA02_H10",
    "B2449500127_n01_LICRNA04_H02",
    "B2449500127_n01_LICRNA04_H11",
]
all_unusable = borderline_unusable + unusable_samples
transcriptome_1_4 = transcriptome_1_4.drop(index=all_unusable)
metadata_1_4 = metadata_1_4.drop(index=all_unusable)
long_term_transcriptome_1_4 = transcriptome_1_4.loc[
    metadata_1_4["Experiment Type"] == "Long Term"
]
ham16s_by_filename_1_4 = metadata_1_4.loc[
    metadata_1_4["Experiment Type"] == "Long Term", "sampID"
].map(total_load_df["ham16s"])

_, go_over_1_4, flg_over_1_4 = load_correlated_genes(
    long_term_transcriptome_1_4, ham16s_by_filename_1_4
)
union_over_0_4_load_1_4 = go_over_1_4 | flg_over_1_4

print(f"Plates 1-4: FLG {len(flg_over_1_4)}, GO {len(go_over_1_4)}, union {len(union_over_0_4_load_1_4)}")
print(f"Plates 1-5: FLG {len(flg_genes_over_0_4_load)}, GO {len(go_term_genes_over_0_4_load)}, union {len(union_over_0_4_load)}")
print(f"\nIn both lists: {len(union_over_0_4_load & union_over_0_4_load_1_4)}")
print(f"Only plates 1-4: {len(union_over_0_4_load_1_4 - union_over_0_4_load)}")
print(f"Only plates 1-5: {len(union_over_0_4_load - union_over_0_4_load_1_4)}")

/Users/michael/miniconda3/envs/Luke_terrace/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/michael/miniconda3/envs/Luke_terrace/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Plates 1-4: FLG 92, GO 120, union 199
Plates 1-5: FLG 63, GO 96, union 148

In both lists: 109
Only plates 1-4: 90
Only plates 1-5: 39


In [10]:
# Plates 1-4 vs plates 1-5 Pearson r for every gene in either list
genes_either = sorted(union_over_0_4_load | union_over_0_4_load_1_4)
pearson_1_4 = long_term_transcriptome_1_4[genes_either].corrwith(ham16s_by_filename_1_4)
gene_list_comparison = pd.DataFrame(
    {
        "pearson_r_plates_1_4": pearson_1_4,
        "pearson_r_plates_1_5": pearson_correlation_with_load.reindex(genes_either),
        "in_plates_1_4_list": [g in union_over_0_4_load_1_4 for g in genes_either],
        "in_plates_1_5_list": [g in union_over_0_4_load for g in genes_either],
    }
)
gene_list_comparison.loc[
    gene_list_comparison["in_plates_1_4_list"] != gene_list_comparison["in_plates_1_5_list"]
].sort_values("pearson_r_plates_1_5", ascending=False)

,pearson_r_plates_1_4,pearson_r_plates_1_5,in_plates_1_4_list,in_plates_1_5_list
AT5G36890,0.235339,0.506470,False,True
AT2G17220,0.392242,0.489924,False,True
AT1G22882,0.339072,0.478459,False,True
AT4G16141,0.391659,0.475263,False,True
AT2G19860,0.351168,0.471666,False,True
...,...,...,...,...
AT5G65200,0.402899,0.214814,True,False
AT5G42810,0.488061,0.211853,True,False
AT1G67060,0.413807,0.196521,True,False
AT1G42540,0.533284,0.194703,True,False


In [11]:
# Check the rebuilt plates 1-4 list against the saved gene list used downstream
saved_defense_genes = set(
    pd.read_csv(
        "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/load_correlated_defense_genes_union_GO_and_FLG.csv"
    )["gene"]
)
print(f"Saved list: {len(saved_defense_genes)} genes")
print(f"Identical to rebuilt plates 1-4 list: {saved_defense_genes == union_over_0_4_load_1_4}")
print(f"Saved list genes also in new plates 1-5 list: {len(saved_defense_genes & union_over_0_4_load)}")

Saved list: 199 genes
Identical to rebuilt plates 1-4 list: True
Saved list genes also in new plates 1-5 list: 109
